# Оценки к опыту «горячая десорбция H₂/D₂ с V(O,S)» — проверка детального равновесия
Записка: `lit_review/H_V_hot_desorption_experiment_proposal_2026-11-09.md`. Первоисточники в `hydrogen_metal_iteraction/ref/`: `eibl_winkler2002.pdf`, `gleispach_winkler2003.pdf`.
Метки: **[данные]** — из статьи; **[память]** — порядок величины не проверен по первоисточнику; **[оценка]** — расчёт здесь.

In [ ]:
preparser(False)
import numpy as np
trapz=getattr(np,'trapz',None) or np.trapezoid
kB=1.380649e-23; amu=1.66054e-27; eV=1.602177e-19; NA=6.02214e23
def check(name, val, ref, rtol=1e-3):
    r=abs(val-ref)/max(abs(ref),1e-300); print(('OK  ' if r<rtol else 'FAIL'), name, val, 'vs', ref, 'rel=%.1e'%r)

## 1. Числа Eibl–Winkler (2002) [данные]
T_s = 950 K. Гипертермальная часть описана максвеллианом потока с T_hyper: 3950 K (S), 2750 K (O). ⟨E⟩ = 2kT_hyper = 0.68 эВ и 0.47 эВ, или 8.3 и 5.8 kT_s.
Прилипание молекулярного D₂ к V(111)+O/S: s₀ ≈ 2·10⁻⁴. Прилипание атомов 0.1–0.2. Аккомодация энергии рассеянных α = 0.37, рассеянные при 950 K ≈ максвеллиан 540 K.
Модель авторов: «дырки» в ячейке без барьера (тепловой канал) + участки с барьером 0.3–0.4 эВ (гипертермальный канал). Через эти участки возможна только адсорбция, на остальной площади молекулы отражаются. Малая доля ~2·10⁻⁴ адсорбирующихся **связана детальным равновесием** с десорбирующимися, то есть авторы детальное равновесие *предполагают*.

In [ ]:
k=kB/eV; Ts=950
for lab,Th,Eref,nref in [('S',3950,0.68,8.3),('O',2750,0.47,5.8)]:
    E=2*k*Th; check('<E> '+lab+' eV',E,Eref,rtol=1e-2); check('<E>/kTs '+lab,E/(k*Ts),nref,rtol=1e-2)

## 2. Почему горячий канал и малое прилипание совместимы с детальным равновесием (проверка внутренней согласованности)
Если прилипание имеет порог E₀ (ступенька, нормальная энергия), то при детальном равновесии поток десорбции = максвелловский поток × S(E). Средняя энергия потока выше порога (вес E·e^{−E/kT}): ⟨E⟩ = (E₀²+2E₀kT+2k²T²)/(E₀+kT) — не E₀+2kT; E₀ находим численно. Интегральное прилипание теплового газа ⟨S⟩ = (1+E₀/kT)e^{−E₀/kT} (для ступеньки S=1 выше порога).
Проверка: какой порог даёт ⟨E⟩ = 0.68 эВ, и каким тогда было бы тепловое прилипание. Сравниваем с измеренным s₀ ≈ 2·10⁻⁴ (оно мерилось при 300 K газа, поэтому сравнение только по порядку).

In [ ]:
from math import exp
def flux_mean_E(E0,T,step=True):
    # numerical: flux ~ E exp(-E/kT) S(E), S=step
    kT=k*T; E=np.linspace(0,40*kT,400001); w=E*np.exp(-E/kT)*(E>=E0)
    return trapz(E*w,E)/trapz(w,E), trapz(w,E)/trapz(E*np.exp(-E/kT),E)
for lab,Eh in [('S',0.68),('O',0.47)]:
    kT=k*Ts; f=lambda E0:(E0**2+2*E0*kT+2*kT**2)/(E0+kT)-Eh
    lo,hi=0.0,Eh
    for _ in range(200):
        mid=(lo+hi)/2; lo,hi=(mid,hi) if f(mid)<0 else (lo,mid)
    E0=lo
    m,Sav=flux_mean_E(E0,Ts)
    check('<E> step model '+lab,m,Eh,rtol=1e-3)
    check('<S> analytic vs numeric '+lab,Sav,(1+E0/(k*Ts))*exp(-E0/(k*Ts)),rtol=1e-3)
    print('   E0=%.3f eV, <S>(T=950K) for hyper channel = %.2e ; at Tgas=300K: %.2e'%(E0,Sav,(1+E0/(k*300))*exp(-E0/(k*300))))

**Вывод [оценка].** Порог ≈0.6 эВ (S) и ≈0.4 эВ (O) (см. вывод ячейки) — порядок 0.3–0.4 эВ авторов; ступенька — грубая модель, у авторов распределение барьеров. Тепловое прилипание газа при 950 K через гипертермальный канал тогда ~10⁻²–10⁻¹·(доля площади), при 300 K — ничтожно. С измеренным s₀ ≈ 2·10⁻⁴ (газ 300 K, идёт в основном через «дырки») это не противоречит. **Значит, данные Eibl–Winkler сами по себе детального равновесия не нарушают и не подтверждают.** Нужна проверка прилипания при T_газа = T_s = 950 K. Это и есть пробел, который закрывает опыт.

## 3. Связь W с нарушением детального равновесия
Мембрана, одна объёмная концентрация c. Сторона i: адсорбция zα_iP_i, десорбция k_i c². Стационарность каждой стороны при нулевом потоке: zα_AP_A = k_Ac², zα_BP_B = k_Bc² ⇒ W = P_B/P_A = (k_B α_A)/(k_A α_B).
Пусть на стороне A (горячая) доля φ теплового канала десорбции подавлена без изменения прилипания: k_A = (1−φ)zα_A/S². На стороне B — детальное равновесие. Тогда W = 1/(1−φ).

In [ ]:
for phi in (1e-3,0.1,0.5):
    z=1.0;S=1.0;aA=2e-4;aB=0.3
    kA=(1-phi)*z*aA/S**2; kB_=z*aB/S**2
    W=(kB_*aA)/(kA*aB); check('W(phi=%g)'%phi,W,1/(1-phi))
# detailed balance both sides
check('W=1 at phi=0',(z*aB*aA)/(z*aA*aB),1.0)

## 4. Этап 1: волюметрия — сколько водорода уходит в образец
Закон Сивертса: x = H/V = K_s√P. [память] K_s для V при ~900 K: H/V ~ 10⁻² при 1 бар (порядок; уточнить по Veleckis–Edwards 1969). Беру K_s = 10⁻²/√(10⁵ Па) и варьирую ×(1/3…3).
Сигнал: различие кажущихся S на δ ⇒ различие поглощённого количества ≈ δ·n_abs.

In [ ]:
Ks0=1e-2/np.sqrt(1e5)
mV=1.0e-3; MV=50.94e-3; nV=mV/MV   # 1 г V
Vgas=1e-3; T=900
for fac in (1/3,1,3):
  for P in (10,100):
    nabs=fac*Ks0*np.sqrt(P)*nV/2      # моль H2
    ngas=P*Vgas/(8.314*T)
    print('Ks x%.2f P=%4d Pa: n_abs=%.2e mol H2, n_gas=%.2e mol, n_abs/n_gas=%.2f, 10%% эффект -> dP/P=%.1e'%(fac,P,nabs,ngas,nabs/ngas,0.1*nabs/ngas))
check('Sieverts sqrt scaling',(Ks0*np.sqrt(100))/(Ks0*np.sqrt(25)),2.0)

**Вывод [оценка].** При 1 г V в 1 л поглощённое количество сравнимо с газовым (n_abs/n_gas ~ 0.1–1). Разница кажущихся растворимостей в 10 % даёт δP/P ~ 10⁻³–10⁻², при разрешении ёмкостного манометра ~10⁻⁴ — с запасом. Для чувствительности нужен объём газа меньше (0.1 л) или образец больше.

## 5. Этап 3: время выравнивания замкнутой ячейки
z = 1/√(2πmkT) (молекул м⁻² с⁻¹ Па⁻¹). Потоком управляет меньшее прилипание α_A. Времена: поверхностное τ_s ≈ V/(z α_A A kT) (для газа) плюс диффузия по толщине L²/D. [память] D(H в V, 900 K) ~ 5·10⁻⁸ м²/с.

In [ ]:
for name,m in (('H2',2.016),('D2',4.028)):
    zz=1/np.sqrt(2*np.pi*m*amu*kB*950); print(name,'z(950K)=%.2e m^-2 s^-1 Pa^-1'%zz)
zH=1/np.sqrt(2*np.pi*2.016*amu*kB*950)
check('z(H2,950)',zH,6.02e22,rtol=5e-3)
A=1e-4; V=1e-3; P=10
Ngas=P*V/(kB*950)
for aA in (1e-2,2e-4,1e-6):
    flux=zH*aA*A*P; tau_s=Ngas/flux
    print('alpha_A=%.0e: flux=%.1e /s, tau_s=%.2e s'%(aA,flux,tau_s))
L=0.5e-3; D=5e-8; print('tau_diff=L^2/D=%.1f s'%(L**2/D))

**Вывод [оценка].** Время ограничено прилипанием на горячей стороне (A = 1 см², V = 1 л, 10 Па). При α ~ 10⁻² ≈1 с, при α ~ 2·10⁻⁴ ≈1 мин, при α ~ 10⁻⁶ ≈3.5 ч. Диффузия ~5 с. Для малых α — увеличить площадь (10 см²) или уменьшить объём (0.1 л). Течь < 10⁻¹⁰ Па·м³/с.
**Поправка к записке:** там τ ~0.1 с при α = 10⁻³ — пересчёт даёт ~13 с. Ошибка ×100 исправлена.

## 6. Тепловая транспирация (манометры при 300 K, ячейка 950 K)
Свободно-молекулярный предел: P_ячейки/P_манометра = √(T_яч/T_ман) = 1.78. Континуальный: 1. Между ними — переходный режим. Знак ошибки одинаков с обеих сторон, поэтому в отношении P_B/P_A ошибка сокращается при одинаковых трубках. Остаётся разница из-за разных давлений и разброса диаметров.
Модель перехода [оценка]: R(Kn) = 1 + (√(T₁/T₂)−1)·Kn/(Kn+0.3), Kn = λ/d, λ = kT/(√2 π d_m² P), d_m(H₂) = 2.9·10⁻¹⁰ м, средняя T трубки.

In [ ]:
def R(P,d,T1=950,T2=300,dm=2.9e-10):
    Tm=0.5*(T1+T2); lam=kB*Tm/(np.sqrt(2)*np.pi*dm**2*P); Kn=lam/d
    return 1+(np.sqrt(T1/T2)-1)*Kn/(Kn+0.3), lam
for P in (1,10,100,1000):
    r,lam=R(P,20e-3)
    rA,_=R(P,20e-3); rB,_=R(P*1.0,20.4e-3)   # 2% разница диаметров
    rW,_=R(P*2,20e-3)                        # если W=2
    print('P=%5d Pa lam=%.2e m R=%.4f  ошибка W от 2%% разницы d: %.1e ; от P_B=2P_A: %.1e'%(P,lam,r,rA/rB-1,rW/r-1))
check('free-molecular limit',R(1e-6,20e-3)[0],np.sqrt(950/300),rtol=1e-4)

**Вывод [оценка].** При трубках 20 мм с разбросом диаметров 2 %: 1000 Па → ошибка W ≈ 6·10⁻⁵; 100 Па → 5·10⁻⁴; 1–10 Па → 2·10⁻³. Значит, рабочий диапазон ≥ 1000 Па (или нуль-манометр внутри печи), плюс перестановка манометров. Если W ≠ 1 по-настоящему, транспирация добавит к нему нелинейную поправку (столбец «от P_B=2P_A»), её надо учитывать калибровкой.

## 7. Изотопный признак дифракции
Длина волны де Бройля при ⟨E⟩ потока 2kT: λ = h/√(2m·2kT). Брэгговский масштаб решётки V(111): межатомное расстояние в плоскости a√2 = 0.428 нм (a = 0.3024 нм). Фасеты (433) крупнее.

In [ ]:
h=6.62607e-34
for name,m in (('H2',2.016),('D2',4.028),('HD',3.022)):
    for T in (300,950):
        lam=h/np.sqrt(2*m*amu*2*kB*T); print('%s T=%d K: lambda=%.3f nm'%(name,T,lam*1e9))
lH=h/np.sqrt(2*2.016*amu*2*kB*950); lD=h/np.sqrt(2*4.028*amu*2*kB*950)
check('lambda_H2/lambda_D2',lH/lD,np.sqrt(4.028/2.016))

**Вывод [оценка].** При 950 K λ ≈ 0.04–0.06 нм, это ~1/10 межатомного расстояния. Дифракционные порядки густые, и при тепловом разбросе энергий (ΔE ~ kT) интерференционные минимумы размываются. Поэтому дифракционный механизм у О.Д. сильнее всего при **низких** T, λ ~ 0.1–0.2 нм. Предложение: дополнить этап 3 точками при 500–600 K, где V ещё проницаем, и сравнить H₂, HD, D₂.

## Сводка
| величина | значение | статус |
|---|---|---|
| ⟨E⟩ гипертерм. | 0.68 / 0.47 эВ (S/O) | данные |
| порог (ступенька), совместимый с ⟨E⟩ | см. ячейку 2 | оценка, порядок 0.3–0.4 эВ авторов |
| s₀(D₂), газ 300 K | 2·10⁻⁴ | данные |
| сигнал этапа 1 (δS = 10 %) | δP/P 10⁻³–10⁻² | оценка, K_s по памяти |
| транспирация, 1000 Па | ~6·10⁻⁵ на W | оценка, модель перехода |
| τ ячейки (α=2·10⁻⁴, 1 см², 1 л) | ~1 мин | оценка |
| λ де Бройля, 950 K | 0.04–0.06 нм | расчёт |